# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import BaseMessage
from lib.tooling import tool

# Optional: use Tavily for web search
from tavily import TavilyClient


In [ ]:
# Load environment variables
# Explicitly pass dotenv_path to handle VS Code local environments where the
# kernel cwd may be the workspace root rather than this notebook's folder.
from pathlib import Path
_env_path = next(
    (p for p in [Path('.env'), Path('project/starter/.env')] if p.exists()),
    Path('.env')
)
load_dotenv(dotenv_path=_env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE")

assert OPENAI_API_KEY, "OPENAI_API_KEY is required"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY is required"
assert TAVILY_API_KEY, "TAVILY_API_KEY is required"

# Debug: Print OpenAI configuration
print("=== OpenAI Configuration ===")
print(f"OPENAI_API_KEY starts with: {OPENAI_API_KEY[:20] if OPENAI_API_KEY else 'NOT SET'}...")
print(f"OPENAI_API_BASE: {OPENAI_API_BASE}")
print()

import openai
openai.api_key = OPENAI_API_KEY
openai.base_url = OPENAI_API_BASE or openai.base_url

print("=== After OpenAI Module Config ===")
print(f"openai.api_key starts with: {openai.api_key[:20] if openai.api_key else 'NOT SET'}...")
print(f"openai.base_url: {openai.base_url}")

In [ ]:
# Instantiate ChromaDB client and get the collection used in Part 1
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    api_base=OPENAI_API_BASE
)

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print(f"Loaded collection: {collection.name}")


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """Search the vector database to retrieve game info relevant to the query.

    Args:
        query: A user question about games.
        n_results: Number of results to return.

    Returns:
        A list of dictionaries containing the retrieved documents and metadata.
    """
    # Normalize query: the LLM may pass a non-string type (list, dict, None)
    # as the query argument. Convert to string and bail out if empty.
    if not isinstance(query, str):
        query = str(query)
    query = query.strip()
    if not query:
        return []

    results = collection.query(query_texts=[query], n_results=n_results)

    docs = []
    for i, doc in enumerate(results.get("documents", [[]])[0]):
        metadata = results.get("metadatas", [[]])[0][i] if results.get("metadatas") else {}
        doc_id = results.get("ids", [[]])[0][i] if results.get("ids") else None
        docs.append({
            "id": doc_id,
            "content": doc,
            "metadata": metadata,
        })

    return docs

#### Evaluate Retrieval Tool

In [ ]:
from pydantic import BaseModel, Field


class RetrievalEvaluation(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents provide enough information to answer the question")
    description: str = Field(description="Explanation of the evaluation decision")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """Evaluate whether the retrieved documents are sufficient to answer the user's question."""

    # The agent may pass retrieved_docs as a JSON string (from tool call serialization),
    # or the list elements may be plain strings rather than dicts.
    if isinstance(retrieved_docs, str):
        try:
            retrieved_docs = json.loads(retrieved_docs)
        except Exception:
            # Keep the original string if parsing fails
            pass

    # Normalize to a list for consistent processing
    if isinstance(retrieved_docs, dict):
        # if a single document was passed as a dict
        retrieved_docs = [retrieved_docs]

    if not isinstance(retrieved_docs, list):
        retrieved_docs = [retrieved_docs] if retrieved_docs else []

    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No documents were retrieved from the local knowledge base.",
        }

    # Simple heuristic: if the query terms appear in any retrieved document metadata or content
    query_tokens = set([t.lower() for t in question.split() if len(t) > 3])

    for doc in retrieved_docs:
        if isinstance(doc, dict):
            content = str(doc.get("content", ""))
            metadata = json.dumps(doc.get("metadata", {}))
        else:
            content = str(doc)
            metadata = ""

        text = " ".join([content, metadata]).lower()
        if any(token in text for token in query_tokens):
            return {
                "useful": True,
                "description": "Retrieved documents appear to contain matching information for the question.",
            }

    return {
        "useful": False,
        "description": "Retrieved documents do not appear to match the question closely enough.",
    }


#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str, max_results: int = 3) -> dict:
    """Search the web using Tavily and return a summarized response."""

    client = TavilyClient(api_key=TAVILY_API_KEY)
    search_result = client.search(
        query=question,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
    )

    return {
        "answer": search_result.get("answer"),
        "results": search_result.get("results", [])[:max_results],
        "search_metadata": {
            "query": question,
            "timestamp": __import__("datetime").datetime.now().isoformat(),
        },
    }


### Agent

In [ ]:
instructions = (
    "You are UdaPlay, an AI research assistant specialized in video games. "
    "Use the provided tools to answer the user's question. "
    "First, attempt to retrieve information using `retrieve_game`. "
    "Then, use `evaluate_retrieval` to decide whether the retrieved information is sufficient. "
    "If the local knowledge is not sufficient, use `game_web_search` to look up information online. "
    "Always provide a clear final answer and cite your sources (e.g., local database, web search)."
)

tools = [retrieve_game, evaluate_retrieval, game_web_search]

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
)

print("Agent initialized with tools:", [t.name for t in tools])


### Reporting

The `print_messages` helper prints the full message trace for each agent run — including the agent's reasoning (tool call decisions) and tool usage results — followed by the final answer. This satisfies the rubric requirement that output shows the agent's reasoning, tool usage, and final answer.

In [ ]:
from lib.messages import BaseMessage

def print_messages(messages: list[BaseMessage]):
    """Print the full agent message trace in a human-readable format.

    Shows each step: system instructions, user query, agent tool call decisions,
    tool results, and the final answer — making the agent's reasoning visible.
    """
    for m in messages:
        tool_calls = getattr(m, 'tool_calls', None)
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {tool_calls})")

In [ ]:
queries = [
    "Which games are set in a fantasy world and involve dragons?",
    "Which gaming platform has a nostalgic collection of classic console games?",
    "Which games are still actively developed and have periodic releases?",
    "Recommend games that are immersive and have strong storytelling elements.",
]

previous_message_count = 0
for query in queries:
    print("=" * 60)
    print(f"Query: {query}")
    print()
    run = agent.invoke(query)
    final_state = run.get_final_state()
    messages = final_state["messages"]

    # Slice to only the messages added during this invocation,
    # excluding the accumulated history carried in from prior queries.
    new_messages = messages[previous_message_count:]
    previous_message_count = len(messages)

    print("Agent Reasoning & Tool Usage:")
    print_messages(new_messages)
    print()
    print("Answer:")
    print(messages[-1].content)
    print()

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes